# Team FSH CV Pipeline

## To-Do
- upload/select video file and turn it into a folder of images
- choose points to track
    - display the 0th frame and allow user to click on specific spots
    - save coordinates of clicked locations for use
    - assign each point to a specific object, and whether the point is inclusion or exclusion
- show results of selected points
- run training
- produce a video output

- mark down frames where number of objects detected drops

> jupyter notebook --no-browser --port=8889 --ip=0.0.0.0

### Configuration

**Global Variables**

In [1]:
INPUT_VID_PATH = "./circle002_30s_1.avi"
JPEGS_DIR_PATH = "./jpegs_dir/" # Must be created ahead before running later code blocks
OUTPUT_PATH = "./output" # Must be created ahead of time

**Model Selection**

In [ ]:
sam2_checkpoint = "../programs/sam2/checkpoints/sam2.1_hiera_large.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"

### Imports

In [6]:
import os
# if using Apple MPS, fall back to CPU for unsupported ops
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image

### Hardware Checks

**Cluster GPU**

The graphic below should show *NVIDIA RTX A4000* or *NVIDIA RTX A5000*. If not, run one of the following commands in the terminal to request a session from the cluster.

> srun --pty --ntasks=4 --gres gpu:rtxa4000:1 --qos scavenger --account scavenger --partition scavenger --mem 20G --time 48:00:00 bash
>
or
>
> srun --pty --ntasks=4 --gres gpu:rtxa5000:1 --qos scavenger --account scavenger --partition scavenger --mem 20G --time 48:00:00 bash

In [7]:
!nvidia-smi

Mon Jun 23 16:01:19 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.153.02             Driver Version: 570.153.02     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A4000               Off |   00000000:22:00.0 Off |                  Off |
| 41%   42C    P8             20W /  140W |       1MiB /  16376MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**PyTorch Settings**

If a GPU has been requested, the following code block should output "cuda". Otherwise, it may output "mps" for Mac chips, or "cpu" for other CPUs. Using a GPU is recommended for timely predictions.

In [8]:
# select the device for computation
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"using device: {device}")

if device.type == "cuda":
    # use bfloat16 for the entire notebook
    torch.autocast("cuda", dtype=torch.float16).__enter__()
    # turn on tfloat32 for Ampere GPUs (https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices)
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
elif device.type == "mps":
    print(
        "\nSupport for MPS devices is preliminary. SAM 2 is trained with CUDA and might "
        "give numerically different outputs and sometimes degraded performance on MPS. "
        "See e.g. https://github.com/pytorch/pytorch/issues/84936 for a discussion."
    )

using device: cuda


### Data Preparation

**Video to JPEG Conversion**

When dealing with video data, the video needs to be converted into a folder of jpegs in order for SAM 2 to process it. Each jpeg in the folder represents a single frame in the original video.

In [24]:
!module load ffmpeg
!ffmpeg -i {INPUT_VID_PATH} -q:v 2 -start_number 0 {JPEGS_DIR_PATH}/'%05d.jpg'

ffmpeg version 7.1 Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 8 (GCC)
  configuration: --prefix=/opt/local/stow/ffmpeg-7.1 --enable-shared --enable-gpl --enable-libass --enable-libfdk-aac --enable-libopus --enable-libtheora --enable-libvorbis --enable-libvpx --enable-libx264 --enable-nonfree --enable-rpath --enable-libx265 --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libfribidi --enable-ffnvcodec
  libavutil      59. 39.100 / 59. 39.100
  libavcodec     61. 19.100 / 61. 19.100
  libavformat    61.  7.100 / 61.  7.100
  libavdevice    61.  3.100 / 61.  3.100
  libavfilter    10.  4.100 / 10.  4.100
  libswscale      8.  3.100 /  8.  3.100
  libswresample   5.  3.100 /  5.  3.100
  libpostproc    58.  3.100 / 58.  3.100
Input #0, avi, from './circle002_30s_1.avi':
  Metadata:
    software        : Lavf61.7.100
  Duration: 00:00:30.00, start: 0.000000, bitrate: 212 kb/s
  Stream #0:0: Video: h264 (High) (H264 / 0x34363248), yuv420p(tv, bt70

**Scanning JPEG Directory**

In [10]:
frame_names = [
    p for p in os.listdir(JPEGS_DIR_PATH)
    if os.path.splitext(p)[-1] in [".jpg", ".jpeg", ".JPG", ".JPEG"]
]
frame_names.sort(key=lambda p: int(os.path.splitext(p)[0]))

### SAM 2 Stuff

**Helper Functions**

In [11]:
def show_mask(mask, ax, obj_id=None, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        cmap = plt.get_cmap("tab10")
        cmap_idx = 0 if obj_id is None else obj_id
        color = np.array([*cmap(cmap_idx)[:3], 0.6])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)

def show_points(coords, labels, ax, marker_size=200):
    pos_points = coords[labels==1]
    neg_points = coords[labels==0]
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)

def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='green', facecolor=(0, 0, 0, 0), lw=2))

**Initializing the Predictor**

In [12]:
from sam2.build_sam import build_sam2_video_predictor
predictor = build_sam2_video_predictor(model_cfg, sam2_checkpoint, device=device)
inference_state = predictor.init_state(
    video_path=JPEGS_DIR_PATH,
    offload_video_to_cpu=True,
    offload_state_to_cpu=False,
    )

frame loading (JPEG): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 800/800 [00:34<00:00, 23.25it/s]


In [13]:
predictor.reset_state(inference_state)

**UI Helper Functions**

In [ ]:
def add_point(frame, obj, points_arr, labels_arr, display):
    points = np.array(points_arr, dtype=np.float32)
    labels = np.array(labels_arr, np.int32)
    # prompts[obj] = points, labels
    _, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
        inference_state = inference_state,
        frame_idx = frame,
        obj_id = obj,
        points = points,
        labels = labels,
    )

    if display:
        plt.figure(figsize = (9, 6))
        plt.title(f"frame {frame}")
        plt.imshow(Image.open(os.path.join(JPEGS_DIR_PATH, frame_names[frame])))
        show_points(points, labels, plt.gca())
        for i, out_obj_id in enumerate(out_obj_ids):
            # show_points(*prompts[out_obj_id], plt.gca())
            show_mask((out_mask_logits[i] > 0.0).cpu().numpy(), plt.gca(), obj_id = out_obj_id)
        plt.savefig(f"{OUTPUT_PATH}/point_add.png")
        plt.close()

def propagate(vis_frame_stride, display):
    # run propagation throughout the video and collect the results in a dict
    video_segments = {}  # video_segments contains the per-frame segmentation results
    for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
        video_segments[out_frame_idx] = {
            out_obj_id: (out_mask_logits[i] > 0.0).cpu().numpy()
            for i, out_obj_id in enumerate(out_obj_ids)
        }

    # render the segmentation results every few frames
    plt.close("all")
    if display:
        for out_frame_idx in range(0, len(frame_names), vis_frame_stride):
            plt.figure(figsize=(6, 4))
            plt.title(f"frame {out_frame_idx}")
            # plt.axis("off")
            plt.imshow(Image.open(os.path.join(JPEGS_DIR_PATH, frame_names[out_frame_idx])))
            for out_obj_id, out_mask in video_segments[out_frame_idx].items():
                show_mask(out_mask, plt.gca(), obj_id=out_obj_id)
            
            out_frame_idx = f"{out_frame_idx:05d}"
            plt.savefig(f"{OUTPUT_PATH}/propagate_{out_frame_idx}.png")
            plt.close()

    return video_segments

### Running Predictions

In [23]:
add_point(0, 1, [[475, 135]], [1], False)
add_point(0, 2, [[330, 600]], [1], False)
vid_seg = propagate(80, True)

propagate in video: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 800/800 [01:52<00:00,  7.13it/s]


**Creating Output Video**

In [16]:
frames = 800
time = 30
fps = frames / time

In [20]:
!ffmpeg -framerate {fps} -start_number 0 -i {OUTPUT_PATH}/"propagate_%05d.png" -c:v libx264 -pix_fmt yuv420p output.mp4

ffmpeg version 7.1 Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 8 (GCC)
  configuration: --prefix=/opt/local/stow/ffmpeg-7.1 --enable-shared --enable-gpl --enable-libass --enable-libfdk-aac --enable-libopus --enable-libtheora --enable-libvorbis --enable-libvpx --enable-libx264 --enable-nonfree --enable-rpath --enable-libx265 --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libfribidi --enable-ffnvcodec
  libavutil      59. 39.100 / 59. 39.100
  libavcodec     61. 19.100 / 61. 19.100
  libavformat    61.  7.100 / 61.  7.100
  libavdevice    61.  3.100 / 61.  3.100
  libavfilter    10.  4.100 / 10.  4.100
  libswscale      8.  3.100 /  8.  3.100
  libswresample   5.  3.100 /  5.  3.100
  libpostproc    58.  3.100 / 58.  3.100
Input #0, image2, from './output/propagate_%05d.png':
  Duration: 00:00:30.00, start: 0.000000, bitrate: N/A
  Stream #0:0: Video: png, rgba(pc, gbr/unknown/unknown), 600x400 [SAR 3937:3937 DAR 3:2], 26.67 fps, 26.67 tbr, 2